# crypto_1m_strat_003

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (76).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `crypto` |
| Trading style | `unknown` |
| Timeframe | `1m` |
| Code cells | 21 |
| Detected functions | fetch_ohlcv_okx_safe, fetch_funding_okx_safe, merge_funding, fetch_ohlcv_okx_spot_safe, backtest_core, calculate_metrics, optimize_worker, optimize, plot_results, print_summary, run_final_backtest |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ccxt

In [ ]:
import ccxt
import pandas as pd
import time

exchange = ccxt.okx({
    'enableRateLimit': True,
    'timeout': 30000,   # 30 seconds
    'options': {
        'defaultType': 'swap'
    }
})


In [ ]:
def fetch_ohlcv_okx_safe(symbol, timeframe='1m', start='2025-11-01'):
    since = exchange.parse8601(start + 'T00:00:00Z')
    all_data = []
    batch_count = 0

    while True:
        try:
            candles = exchange.fetch_ohlcv(
                symbol=symbol,
                timeframe=timeframe,
                since=since,
                limit=100
            )

            if not candles:
                break

            all_data.extend(candles)
            since = candles[-1][0] + 1
            batch_count += 1

            if batch_count % 10 == 0:
                print(f"{symbol}: fetched {len(all_data)} candles")

            if len(candles) < 100:
                break

            time.sleep(exchange.rateLimit / 1000)

        except Exception as e:
            print("Retrying:", e)
            time.sleep(2)

    df = pd.DataFrame(
        all_data,
        columns=['timestamp','open','high','low','close','volume']
    )

    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    return df


In [ ]:
def fetch_funding_okx_safe(symbol):
    while True:
        try:
            data = exchange.fetch_funding_rate_history(symbol)
            break
        except Exception as e:
            print("Funding retry:", e)
            time.sleep(2)

    df = pd.DataFrame(data)
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df['funding'] = df['fundingRate'].astype(float)

    return df[['timestamp','funding']]


In [ ]:
def merge_funding(ohlcv, funding):
    df = ohlcv.merge(funding, on='timestamp', how='left')
    df['funding'] = df['funding'].ffill().fillna(0.0)
    return df


In [ ]:
btc_df = merge_funding(
    fetch_ohlcv_okx_safe('BTC-USDT-SWAP'),
    fetch_funding_okx_safe('BTC-USDT-SWAP')
)

eth_df = merge_funding(
    fetch_ohlcv_okx_safe('ETH-USDT-SWAP'),
    fetch_funding_okx_safe('ETH-USDT-SWAP')
)

print(len(btc_df), len(eth_df))
btc_df


In [ ]:
btc_df["ticker"] = "BTC"
eth_df["ticker"] = "ETH"

eth_df

In [ ]:
exchange_spot = ccxt.okx({
    'enableRateLimit': True,
    'timeout': 30000,
    'options': {
        'defaultType': 'spot'
    }
})

In [ ]:
def fetch_ohlcv_okx_spot_safe(symbol, timeframe='1m', start='2025-01-01'):
    since = exchange_spot.parse8601(start + 'T00:00:00Z')
    all_data = []
    batch_count = 0

    while True:
        try:
            candles = exchange_spot.fetch_ohlcv(
                symbol=symbol,
                timeframe=timeframe,
                since=since,
                limit=100
            )

            if not candles:
                break

            all_data.extend(candles)
            since = candles[-1][0] + 1
            batch_count += 1

            if batch_count % 10 == 0:
                print(f"{symbol} (spot): fetched {len(all_data)} candles")

            if len(candles) < 100:
                break

            time.sleep(exchange_spot.rateLimit / 1000)

        except Exception as e:
            print("Retrying:", e)
            time.sleep(2)

    df = pd.DataFrame(
        all_data,
        columns=['timestamp','open','high','low','close','volume']
    )

    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    return df


In [ ]:
# Spot symbols (IMPORTANT)
eth_btc_spot = fetch_ohlcv_okx_spot_safe('ETH/BTC', '1m', '2025-11-01')
eth_btc_spot


In [ ]:
!pip install pykalman

In [ ]:
from pykalman import KalmanFilter
import numpy as np

df = eth_df[['timestamp', 'close', 'funding']].rename(
    columns={'close': 'close_eth', 'funding': 'funding_eth'}
)

df = df.merge(
    btc_df[['timestamp', 'close', 'funding']].rename(
        columns={'close': 'close_btc', 'funding': 'funding_btc'}
    ),
    on='timestamp',
    how='inner'
)

df = df.merge(
    eth_btc_spot[['timestamp', 'close']].rename(
        columns={'close': 'close_ethbtc'}
    ),
    on='timestamp',
    how='inner'
)

df = df.sort_values('timestamp').reset_index(drop=True)

df['log_eth'] = np.log(df['close_eth']).ewm(span=3).mean()
df['log_btc'] = np.log(df['close_btc']).ewm(span=3).mean()
df['log_ethbtc'] = np.log(df['close_ethbtc']).ewm(span=3).mean()

In [ ]:
df['log_perp_ratio'] = df['log_eth'] - df['log_btc']

df['spread_raw'] = df['log_perp_ratio'] - df['log_ethbtc']

kf = KalmanFilter(
    transition_matrices=[1],
    observation_matrices=[1],
    initial_state_mean=0,
    initial_state_covariance=1,
    transition_covariance=1e-4,
    observation_covariance=1e-3
)

state_means, state_covs = kf.filter(df['spread_raw'].values)

df['spread_kf'] = (state_means.flatten())
df['spread_std'] = (np.sqrt(state_covs.flatten()))

df['zscore'] = (df['spread_raw'] - df['spread_kf']) / df['spread_std']

df['funding_diff'] = (df['funding_eth'] - df['funding_btc'])
df

In [ ]:
df.columns

In [ ]:
df.to_csv('/content/drive/MyDrive/btc_eth_prep_ethbtc_spot_data.csv', index=False)

In [ ]:
df - pd.read_csv('/content/drive/MyDrive/btc_eth_prep_ethbtc_spot_data.csv')
df

In [ ]:
import pandas as pd

# Assuming your dataframe is named 'df'
# df = pd.read_csv('your_file.csv')

# Convert timestamp to datetime if it's not already
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Set timestamp as index for resampling
df.set_index('timestamp', inplace=True)

# Define aggregation rules for each column
agg_dict = {
    'close_eth': 'last',        # Last value in the hour
    'funding_eth': 'mean',      # Average funding rate
    'close_btc': 'last',        # Last value in the hour
    'funding_btc': 'mean',      # Average funding rate
    'log_eth': 'last',          # Last value in the hour
    'log_btc': 'last',          # Last value in the hour
    'beta': 'mean',             # Average beta
    'spread': 'last',           # Last spread value
    'spread_mean': 'last',      # Last spread mean
    'spread_std': 'last',       # Last spread std
    'z': 'last',                # Last z-score
    'funding_diff': 'mean',     # Average funding difference
    'effective_z': 'last'       # Last effective z-score
}

# Resample to hourly frequency ('H')
df = df.resample('5min').agg(agg_dict)

# Reset index to make timestamp a column again
df.reset_index(inplace=True)

# Display results
print(f"Original data: {len(df)} rows (minute-level)")
print(f"Hourly data: {len(df)} rows")
print("\nFirst few rows of hourly data:")

df

In [ ]:
Z_ENTER = 0.015
Z_EXIT = 0.003
FUNDING_MIN = 0.000025  # ignore noise funding

df['signal'] = 0

# ETH overpriced vs BTC → short ETH, long BTC
short_eth = (
    (df['zscore'] > Z_ENTER) &
    (df['funding_diff'] > FUNDING_MIN)
)

# ETH underpriced vs BTC → long ETH, short BTC
long_eth = (
    (df['zscore'] < -Z_ENTER) &
    (df['funding_diff'] < -FUNDING_MIN)
)

df.loc[short_eth, 'signal'] = -1
df.loc[long_eth, 'signal'] = 1

exit_cond = (
    (df['zscore'].abs() < Z_EXIT) |
    (df['funding_diff'].abs() < FUNDING_MIN)
)

df.loc[exit_cond, 'signal'] = 0

df['signal'].shift(1)

df['pos_eth_perp'] = df['signal']
df['pos_btc_perp'] = -df['signal']
df['pos_ethbtc_spot'] = df['signal']

df

df[['zscore', 'funding_diff', 'signal']].plot(subplots=True, figsize=(12,6))

In [ ]:
import pandas as pd
import numpy as np

# =====================
# PARAMETERS
# =====================
Z_ENTER = 0.01
Z_EXIT = 0.0001
FUNDING_MIN = 0.000025

NOTIONAL = 10_000
TAKER_FEE = 0.0004
SLIPPAGE = 0.0002
FUNDING_INTERVAL_MIN = 480
MAX_HOLD_MIN = 2880

# =====================
# SIGNAL GENERATION
# =====================
df = df.copy().sort_values('timestamp').reset_index(drop=True)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Generate entry signals
short_eth = (df['zscore'] > Z_ENTER) & (df['funding_diff'] > FUNDING_MIN)
long_eth = (df['zscore'] < -Z_ENTER) & (df['funding_diff'] < -FUNDING_MIN)

# Generate exit signals
exit_cond = (df['zscore'].abs() < Z_EXIT) | (df['funding_diff'].abs() < FUNDING_MIN)

df['signal'] = 0
df.loc[short_eth, 'signal'] = -1
df.loc[long_eth, 'signal'] = 1
df.loc[exit_cond, 'signal'] = 0

# Shift signal to avoid look-ahead bias
df['signal'] = df['signal'].shift(1).fillna(0)

# =====================
# BACKTESTING ENGINE
# =====================
trades = []
pos = 0
entry_idx = None
funding_accum = 0.0

for i in range(1, len(df)):
    row = df.iloc[i]

    # ENTRY
    if pos == 0 and row['signal'] != 0:
        pos = row['signal']
        entry_idx = i
        funding_accum = 0.0

        # Store entry prices with slippage (only perps, no spot)
        entry_eth = row['close_eth'] * (1 + SLIPPAGE * pos)
        entry_btc = row['close_btc'] * (1 - SLIPPAGE * pos)
        entry_time = row['timestamp']

    # HOLD & EXIT
    elif pos != 0:
        bars_held = i - entry_idx

        # Accrue funding every interval
        if bars_held % FUNDING_INTERVAL_MIN == 0:
            funding_accum += -pos * NOTIONAL * row['funding_eth']
            funding_accum += pos * NOTIONAL * row['funding_btc']

        # Check exit conditions
        should_exit = row['signal'] == 0 or bars_held >= MAX_HOLD_MIN

        if should_exit:
            # Calculate exit prices with slippage (only perps, no spot)
            exit_eth = row['close_eth'] * (1 - SLIPPAGE * pos)
            exit_btc = row['close_btc'] * (1 + SLIPPAGE * pos)

            # Calculate PnL components (only perps)
            pnl_eth = pos * NOTIONAL * (exit_eth / entry_eth - 1)
            pnl_btc = -pos * NOTIONAL * (exit_btc / entry_btc - 1)

            fees = NOTIONAL * TAKER_FEE * 4  # 2 entries + 2 exits
            total_pnl = pnl_eth + pnl_btc + funding_accum - fees

            trades.append({
                'entry_time': entry_time,
                'exit_time': row['timestamp'],
                'side': 'LONG_ETH' if pos == 1 else 'SHORT_ETH',
                'hold_min': bars_held,
                'pnl_eth': pnl_eth,
                'pnl_btc': pnl_btc,
                'funding': funding_accum,
                'fees': fees,
                'total_pnl': total_pnl
            })

            pos = 0

# =====================
# RESULTS
# =====================
tradebook = pd.DataFrame(trades)
tradebook['cum_pnl'] = tradebook['total_pnl'].cumsum()

print(f"Total Trades: {len(tradebook)}")
print(f"Win Rate: {(tradebook['total_pnl'] > 0).mean():.2%}")
print(f"Avg Trade PnL: ${tradebook['total_pnl'].mean():.2f}")
print(f"Total PnL: ${tradebook['total_pnl'].sum():.2f}")
print(f"Max Drawdown: ${(tradebook['cum_pnl'] - tradebook['cum_pnl'].cummax()).min():.2f}")

tradebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from numba import jit
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
from itertools import product

# Constants
NOTIONAL_SIZE = 10000
TAKER_FEE = 0.0005

@jit(nopython=True)
def backtest_core(close_eth, close_btc, z, effective_z, beta,
                  entry_z, exit_z, max_z, max_hold, lookback):
    """JIT-compiled backtest engine"""
    n, position, trade_idx = len(close_eth), 0, 0
    entry_idx, entry_eth, entry_btc = 0, 0.0, 0.0
    eth_qty, btc_qty, eth_not, btc_not = 0.0, 0.0, 0.0, 0.0

    trades = np.zeros(n - lookback)
    equity = np.zeros(n - lookback)
    cumulative = 0.0

    for i in range(lookback, n):
        if position == 0:  # Entry
            if abs(effective_z[i]) > entry_z:
                position = 1 if effective_z[i] < 0 else -1
                eth_not, btc_not = NOTIONAL_SIZE, beta[i] * NOTIONAL_SIZE
                eth_qty, btc_qty = eth_not / close_eth[i], btc_not / close_btc[i]
                entry_idx, entry_eth, entry_btc = i, close_eth[i], close_btc[i]

        else:  # Exit check
            exit_cond = (abs(z[i]) < exit_z or abs(z[i]) > max_z or
                        (i - entry_idx) >= max_hold)

            if exit_cond:
                # PnL calculation
                eth_pnl = (close_eth[i] - entry_eth) * eth_qty * position
                btc_pnl = (entry_btc - close_btc[i]) * btc_qty * position
                fees = (eth_not + btc_not + eth_qty * close_eth[i] +
                       btc_qty * close_btc[i]) * TAKER_FEE

                net_pnl = eth_pnl + btc_pnl - fees
                cumulative += net_pnl

                trades[trade_idx] = net_pnl
                equity[trade_idx] = cumulative
                trade_idx += 1
                position = 0

    return trades[:trade_idx], equity[:trade_idx]

def calculate_metrics(trades, equity):
    """Fast vectorized metrics"""
    if len(trades) < 5:
        return None

    wins = trades[trades > 0]
    losses = trades[trades < 0]

    profit_factor = wins.sum() / abs(losses.sum()) if len(losses) > 0 else 0
    max_dd = (equity - np.maximum.accumulate(equity)).min() if len(equity) > 0 else 0

    # Composite score
    score = (trades.mean() +
            (len(wins) / len(trades)) * 1000 +
            profit_factor * 100 -
            max_dd / 100)

    return {
        'avg_pnl': trades.mean(),
        'win_rate': len(wins) / len(trades),
        'profit_factor': profit_factor,
        'max_drawdown': max_dd,
        'total_trades': len(trades),
        'total_pnl': trades.sum(),
        'composite_score': score
    }

def optimize_worker(args):
    """Parallel worker"""
    entry_z, exit_z, max_z, max_hold, data, lookback = args

    if exit_z >= entry_z or max_z <= entry_z:
        return None

    try:
        trades, equity = backtest_core(*data, entry_z, exit_z, max_z, max_hold, lookback)
        metrics = calculate_metrics(trades, equity)

        if metrics:
            return {
                'entry_z': entry_z, 'exit_z': exit_z,
                'max_z': max_z, 'max_hold': max_hold,
                **metrics
            }
    except:
        pass
    return None

def optimize(df, lookback=24):
    """Main optimization with parallel processing"""
    # Parameter grid
    params = list(product(
        np.arange(2.0, 6.0, 0.2),  # entry_z
        np.arange(0.1, 1.0, 0.2),  # exit_z
        np.arange(5.0, 8.0, 0.5),   # max_z
        [70, 80, 90, 100, 110, 120, 130, 140, 150] # max_hold
    ))

    # Extract data once
    data = (df['close_eth'].values, df['close_btc'].values,
            df['z'].values, df['effective_z'].values, df['beta'].values)

    args = [(e, x, m, h, data, lookback) for e, x, m, h in params]

    print(f"Testing {len(params)} combinations on {mp.cpu_count()} cores...")

    results = []
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as executor:
        futures = [executor.submit(optimize_worker, arg) for arg in args]
        for future in tqdm(as_completed(futures), total=len(futures)):
            if result := future.result():
                results.append(result)

    return pd.DataFrame(results)

def plot_results(df):
    """Generate comprehensive sensitivity plots"""
    fig = plt.figure(figsize=(20, 15))

    params_info = [
        ('entry_z', 'Entry Z-Score'),
        ('exit_z', 'Exit Z-Score'),
        ('max_z', 'Max Z-Score (Stop Loss)'),
        ('max_hold', 'Max Hold Period (hours)')
    ]

    metrics_info = [
        ('avg_pnl', 'Average PnL ($)', 'blue'),
        ('win_rate', 'Win Rate', 'green'),
        ('profit_factor', 'Profit Factor', 'orange'),
        ('composite_score', 'Composite Score', 'red')
    ]

    # Create 4x4 grid: 4 parameters × 4 metrics
    for i, (param, param_label) in enumerate(params_info):
        for j, (metric, metric_label, color) in enumerate(metrics_info):
            ax = plt.subplot(4, 4, i * 4 + j + 1)

            # Group and plot
            grouped = df.groupby(param)[metric].mean()
            ax.plot(grouped.index, grouped.values, marker='o', linewidth=2.5,
                   markersize=7, color=color, alpha=0.8)

            # Formatting
            ax.set_xlabel(param_label, fontsize=10, fontweight='bold')
            ax.set_ylabel(metric_label, fontsize=10, fontweight='bold')
            ax.set_title(f'{param_label} vs {metric_label}', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, linestyle='--')

            # Format y-axis for win_rate as percentage
            if metric == 'win_rate':
                ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.1f}%'))
            elif metric == 'avg_pnl':
                ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'${y:.0f}'))

    plt.suptitle('Parameter Sensitivity Analysis: All Metrics',
                 fontsize=18, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig('optimization_results.png', dpi=300, bbox_inches='tight')
    plt.show()

def print_summary(df):
    """Print optimization summary"""
    best = df.loc[df['composite_score'].idxmax()]

    print("\n" + "="*80)
    print("🏆 BEST PARAMETERS")
    print("="*80)
    print(f"Entry Z: {best['entry_z']:.2f} | Exit Z: {best['exit_z']:.2f} | "
          f"Max Z: {best['max_z']:.2f} | Max Hold: {best['max_hold']:.0f}h")
    print(f"\nComposite Score: {best['composite_score']:.2f}")
    print(f"Avg PnL: ${best['avg_pnl']:.2f} | Win Rate: {best['win_rate']*100:.2f}% | "
          f"PF: {best['profit_factor']:.3f}")
    print(f"Max DD: ${best['max_drawdown']:.2f} | Trades: {best['total_trades']:.0f} | "
          f"Total PnL: ${best['total_pnl']:.2f}")

    print("\n" + "="*80)
    print("TOP 5 COMBINATIONS")
    print("="*80)
    top5 = df.nlargest(5, 'composite_score')
    print(top5[['entry_z', 'exit_z', 'max_z', 'max_hold', 'composite_score',
                'avg_pnl', 'win_rate', 'profit_factor', 'total_trades']].to_string(index=False))
    print("="*80)

def run_final_backtest(df, params, lookback=24):
    """Run detailed backtest with best parameters"""
    data = (df['close_eth'].values, df['close_btc'].values,
            df['z'].values, df['effective_z'].values, df['beta'].values)

    trades, equity = backtest_core(
        *data, params['entry_z'], params['exit_z'],
        params['max_z'], int(params['max_hold']), lookback
    )

    return pd.DataFrame({
        'net_pnl': trades,
        'return_pct': (trades / NOTIONAL_SIZE) * 100
    })

# MAIN EXECUTION
if __name__ == "__main__":
    print("Starting optimization...\n")

    results = optimize(df, lookback=120)
    results.to_csv('optimization_results.csv', index=False)

    print(f"\n✅ Tested {len(results)} valid combinations")
    print_summary(results)

    plot_results(results)
    print("\n✅ Plot saved as 'optimization_results.png'")

    # Final backtest
    best = results.loc[results['composite_score'].idxmax()]
    trades = run_final_backtest(df, best, lookback=120)
    trades.to_csv('optimal_tradebook.csv', index=False)
    print(f"✅ Optimal tradebook saved ({len(trades)} trades)")